In [1]:
!pip install ddgs sentence-transformers transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 44.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.1 MB/s eta 0:00:00:00:010:01m


In [2]:
import re
import torch
from ddgs import DDGS
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
class ClaimExtractor:

    def extract_claims(self, article):

        sentences = re.split(r'[.!?]+', article)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

        claims = sentences[:3]

        return claims

class QueryGenerator:

    def generate_queries(self, claims):

        queries = []

        for claim in claims:

            # ✅ FULL CLAIM (VERY IMPORTANT)
            queries.append(claim)

            # ✅ keyword version
            words = re.findall(r'\b[A-Za-z0-9]+\b', claim)
            keywords = [w for w in words if len(w) > 3]

            keyword_query = " ".join(keywords[:8])

            if keyword_query.strip():
                queries.append(keyword_query)

        return list(set(queries))[:5]
        
from ddgs import DDGS
from urllib.parse import urlparse

class KnowledgeRetriever:

    def __init__(self):

        # ✅ Expanded credible domains
        self.credible_domains = {

            # High trust news
            "reuters.com":1.0,
            "apnews.com":1.0,
            "bbc.com":1.0,
            "bbc.co.uk":1.0,
            "wikipedia.org":1.0,
            "britannica.com":0.95,
            "encyclopedia.com":0.9,

            # Major media
            "nytimes.com":0.95,
            "theguardian.com":0.95,
            "washingtonpost.com":0.95,
            "wsj.com":0.95,
            "bloomberg.com":0.95,

            # Fact-checking ⭐
            "snopes.com":1.0,
            "factcheck.org":1.0,
            "politifact.com":1.0,

            # Science / health
            "nature.com":0.95,
            "science.org":0.95,
            "sciencedaily.com":0.9,
            "nih.gov":1.0,
            "who.int":1.0,
            "cdc.gov":1.0,

            # Educational / government
            ".edu":0.9,
            ".gov":1.0
        }

    # ✅ Extract domain cleanly
    def extract_domain(self, url):

        try:
            domain = urlparse(url).netloc.lower()
            domain = domain.replace("www.", "")
            return domain
        except:
            return ""

    # ✅ Assign credibility score
    def get_credibility(self, domain):

        for d, score in self.credible_domains.items():
            if d in domain:
                return score

        return 0.3  # default low

    # ✅ Detect source type (optional but useful)
    def get_source_type(self, domain):

        if ".gov" in domain or ".edu" in domain:
            return "HIGH_TRUST"

        if any(d in domain for d in ["reuters","bbc","apnews"]):
            return "NEWS"

        if any(d in domain for d in ["snopes","factcheck","politifact"]):
            return "FACT_CHECK"

        return "UNKNOWN"

    # ✅ MAIN SEARCH FUNCTION
    def search(self, query):

        evidence = []

        if query.strip() == "":
            return evidence

        try:
            with DDGS() as ddgs:

                results = ddgs.text(query, max_results=10)

                for r in results:

                    url = r.get("href", "")
                    title = r.get("title", "")
                    snippet = r.get("body", "")

                    domain = self.extract_domain(url)

                    credibility = self.get_credibility(domain)

                    # ✅ FILTER LOW QUALITY
                    if credibility < 0.6:
                        continue

                    evidence.append({
                        "title": title,
                        "snippet": snippet,
                        "url": url,
                        "domain": domain,
                        "credibility_score": credibility,
                        "source_type": self.get_source_type(domain)
                    })

        except Exception as e:
            print("Search Error:", e)

        # ✅ REMOVE DUPLICATES
        unique = []
        seen = set()

        for e in evidence:
            if e["url"] not in seen:
                unique.append(e)
                seen.add(e["url"])

        return unique

class EvidenceRanker:

    def __init__(self):
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if len(evidence) == 0:
            return []

        claim_emb = self.encoder.encode(claim, convert_to_tensor=True)

        snippets = [e["snippet"] for e in evidence]
        snippet_emb = self.encoder.encode(snippets, convert_to_tensor=True)

        similarities = util.cos_sim(claim_emb, snippet_emb)[0]

        for i, e in enumerate(evidence):

            semantic = float(similarities[i])
            credibility = e["credibility_score"]

            # ✅ improved weighting
            e["score"] = 0.6 * semantic + 0.4 * credibility

        ranked = sorted(evidence, key=lambda x: x["score"], reverse=True)

        return ranked[:5]
        

class ClaimVerifier:

    def __init__(self):

        model_name = "mistralai/Mistral-7B-Instruct-v0.2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    def verify(self, claim, evidence):

        # ---------- Handle empty evidence ----------
        if len(evidence) == 0:
            return "UNVERIFIED",0.4,"Insufficient evidence"

        # ---------- Prepare evidence ----------
        top_evidence = evidence[:3]
        evidence_text = "\n".join([e["snippet"] for e in top_evidence])

        # ---------- Improved Prompt ----------
        prompt = f"""
You are a fact-checking assistant.

Claim:
{claim}

Evidence:
{evidence_text}

Task:
- Check whether evidence supports or contradicts the claim
- Use only given evidence

Respond EXACTLY like this:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation based on evidence
"""

        # ---------- Tokenize ----------
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # ---------- Generate ----------
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # ---------- Default values ----------
        verdict = "REAL"
        confidence = 0.5
        explanation = "Could not determine clearly."

        # ---------- Safe Parsing ----------
        for line in response.split("\n"):

            line_upper = line.upper()

            if "VERDICT" in line_upper:
                if "FALSE" in line_upper:
                    verdict = "FAKE"
                elif "TRUE" in line_upper:
                    verdict = "REAL"

            elif "CONFIDENCE" in line_upper:
                import re
                nums = re.findall(r"\d*\.?\d+", line)
                if nums:
                    confidence = float(nums[0])

            elif "EXPLANATION" in line_upper:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    explanation = parts[1].strip()

        # ---------- Add Source-Based Explanation ----------
        high_cred_sources = [e for e in evidence if e["credibility_score"] >= 0.9]
        domains = [e["domain"] for e in top_evidence if "domain" in e]

        if verdict == "REAL":
            explanation += f" Supported by {len(high_cred_sources)} high-credibility sources."
        else:
            explanation += f" Contradicted by credible sources."

        if domains:
            explanation += f" Sources include: {', '.join(domains[:2])}."

        return verdict, confidence, explanation

class FakeNewsPipeline:

    def __init__(self):

        self.extractor = ClaimExtractor()
        self.query_gen = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()

    def detect(self, article):

        claims = self.extractor.extract_claims(article)

        if not claims:
            return None

        results = []

        for claim in claims:

            queries = self.query_gen.generate_queries([claim])

            evidence = []

            for q in queries:
                evidence.extend(self.retriever.search(q))

            ranked = self.ranker.rank(claim, evidence)

            verdict, conf, exp = self.verifier.verify(claim, ranked)

            results.append({
                "verdict": verdict,
                "confidence": conf
            })

        # ✅ weighted decision
        fake_score = sum(r["confidence"] for r in results if r["verdict"] == "FAKE")
        real_score = sum(r["confidence"] for r in results if r["verdict"] == "REAL")

        if fake_score > real_score * 1.1:
            final = "FAKE"
        elif real_score > fake_score * 1.1:
            final = "REAL"
        else:
            final = "PARTIALLY FAKE"

        avg_conf = sum(r["confidence"] for r in results) / len(results)

        return {
            "verdict": final,
            "confidence": avg_conf
        }

In [4]:
import pandas as pd

# Load dataset
covid_df = pd.read_csv("/kaggle/input/datasets/thesumitbanik/covid-fake-news-dataset/data.csv")

# Check columns
print(covid_df.columns)
covid_df.head()
print(covid_df.columns)

Index(['headlines', 'outcome'], dtype='object')
Index(['headlines', 'outcome'], dtype='object')


In [5]:
# Select correct columns
covid_df = covid_df[["headlines", "outcome"]]

# Rename to match pipeline
covid_df = covid_df.rename(columns={
    "headlines": "text",
    "outcome": "label"
})

# Convert labels (IMPORTANT)
# Usually: 1 = REAL, 0 = FAKE (verify if needed)
covid_df["label"] = covid_df["label"].astype(int)

# Remove missing values
covid_df = covid_df.dropna()

print("Dataset ready:", covid_df.shape)
covid_df.head()

Dataset ready: (10201, 2)


,text,label
0,A post claims compulsory vacination violates t...,0
1,A photo claims that this person is a doctor wh...,0
2,Post about a video claims that it is a protest...,0
3,All deaths by respiratory failure and pneumoni...,0
4,The dean of the College of Biologists of Euska...,0


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pipeline = FakeNewsPipeline()

y_true = []
y_pred = []

data = covid_df.sample(200, random_state=42)


for _, row in data.iterrows():

    article = row["text"]
    label = "REAL" if row["label"] == 1 else "FAKE"

    try:
        result = pipeline.detect(article, verbose=False)

        # ✅ check result
        if result is None:
            continue

        pred = result["verdict"]

        # ✅ convert PARTIALLY FAKE → FAKE
        if pred == "PARTIALLY FAKE":
            pred = "FAKE"

        # ✅ skip invalid predictions
        if pred not in ["REAL", "FAKE"]:
            continue

    except:
        pred = "FAKE"   # fallback

    y_true.append(label)
    y_pred.append(pred)

# ---------- METRICS ----------
print("\n==============================")
print("MODEL EVALUATION (COVID DATASET)")
print("==============================")

print("Accuracy:", round(accuracy_score(y_true,y_pred),2))
print("Precision:", round(precision_score(y_true,y_pred,pos_label="REAL",zero_division=0),2))
print("Recall:", round(recall_score(y_true,y_pred,pos_label="REAL",zero_division=0),2))
print("F1 Score:", round(f1_score(y_true,y_pred,pos_label="REAL",zero_division=0),2))

print("\nDetailed Report:\n")
print(classification_report(y_true,y_pred,zero_division=0))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Search Error: No results found.

MODEL EVALUATION (COVID DATASET)
Accuracy: 0.86
Precision: 0.15
Recall: 0.5
F1 Score: 0.23

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.98      0.88      0.93       192
        REAL       0.15      0.50      0.23         8

    accuracy                           0.86       200
   macro avg       0.56      0.69      0.58       200
weighted avg       0.94      0.86      0.90       200



In [7]:
import pandas as pd

# Load JSONL file
import json

data = []

with open("/kaggle/input/datasets/sonavp/fever-dataset/train.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

fever_df = pd.DataFrame(data)

# Keep required columns
fever_df = fever_df[["claim", "label"]]

# Remove NEI
fever_df = fever_df[fever_df["label"] != "NOT ENOUGH INFO"]

# Convert labels
def convert_label(x):
    if x == "SUPPORTS":
        return "REAL"
    else:
        return "FAKE"

fever_df["label"] = fever_df["label"].apply(convert_label)

# Rename for pipeline
fever_df = fever_df.rename(columns={"claim": "text"})

print(fever_df.head())
print("Total samples:", len(fever_df))

                                                text label
0                                                     FAKE
1  Michael Folivi competed with ten teams from 20...  FAKE
2  Asiatic Society of Bangladesh(housed in Nimtal...  REAL
3  Lindfield railway station has 3 bus routes, in...  REAL
4  Mukaradeeb('Wolf's Den') is a city in Iraq nea...  REAL
Total samples: 69051


In [8]:
# Use small sample (LLM is slow)
fever_sample = fever_df.sample(400, random_state=42)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
pipeline = FakeNewsPipeline()

y_true = []
y_pred = []

for i, row in fever_sample.iterrows():

    article = row["text"]
    true_label = row["label"]

    try:
        result = pipeline.detect(article)

        if result is None:
            continue

        predicted = result["verdict"]

        # Handle PARTIALLY FAKE → treat as FAKE
        if predicted == "PARTIALLY FAKE":
            predicted = "FAKE"

        # Skip UNVERIFIED
        if predicted not in ["REAL", "FAKE"]:
            continue

        y_true.append(true_label)
        y_pred.append(predicted)

    except Exception as e:
        continue

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average="binary", pos_label="REAL")
recall = recall_score(y_true, y_pred, average="binary", pos_label="REAL")
f1 = f1_score(y_true, y_pred, average="binary", pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION (FEVER)")
print("==============================")

print("Accuracy:", round(accuracy, 2))
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))

print("\nDetailed Report:\n")
print(classification_report(y_true, y_pred))


MODEL EVALUATION (FEVER)
Accuracy: 0.66
Precision: 0.65
Recall: 0.83
F1 Score: 0.73

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.67      0.44      0.53       176
        REAL       0.65      0.83      0.73       224

    accuracy                           0.66       400
   macro avg       0.66      0.63      0.63       400
weighted avg       0.66      0.66      0.64       400



In [3]:
class ClaimExtractor:

    def extract_claims(self, article):

        sentences = re.split(r'[.!?]+', article)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 25]

        claims = sentences[:3]

        return claims

class QueryGenerator:

    def generate_queries(self, claims):

        queries = []

        for claim in claims:

            words = re.findall(r'\b[A-Za-z0-9]+\b', claim)

            keywords = [w for w in words if len(w) > 3]

            queries.append(" ".join(keywords[:8]))

        return queries[:5]
from ddgs import DDGS
from urllib.parse import urlparse

class KnowledgeRetriever:

    def __init__(self):

        # ✅ Expanded credible domains
        self.credible_domains = {

    # =========================
    # ⭐ FACT-CHECKING (HIGHEST)
    # =========================
    "snopes.com":1.0,
    "factcheck.org":1.0,
    "politifact.com":1.0,
    "fullfact.org":1.0,
    "afp.com":1.0,
    "reuters.com":1.0,
    "apnews.com":1.0,

    # =========================
    # 🏛 GOVERNMENT / OFFICIAL
    # =========================
    ".gov":1.0,
    "who.int":1.0,
    "cdc.gov":1.0,
    "nih.gov":1.0,
    "fda.gov":1.0,
    "europa.eu":1.0,
    "gov.uk":1.0,

    # =========================
    # 🎓 EDUCATION
    # =========================
    ".edu":0.95,
    "harvard.edu":0.95,
    "stanford.edu":0.95,
    "mit.edu":0.95,
    "ox.ac.uk":0.95,
    "cam.ac.uk":0.95,

    # =========================
    # 🔬 SCIENTIFIC JOURNALS
    # =========================
    "nature.com":0.95,
    "science.org":0.95,
    "sciencedirect.com":0.9,
    "springer.com":0.9,
    "frontiersin.org":0.9,
    "plos.org":0.9,
    "jamanetwork.com":0.95,
    "thelancet.com":0.95,
    "bmj.com":0.95,

    # =========================
    # 🧬 MEDICAL / HEALTH
    # =========================
    "pubmed.ncbi.nlm.nih.gov":1.0,
    "ncbi.nlm.nih.gov":1.0,
    "mayoclinic.org":0.95,
    "clevelandclinic.org":0.95,
    "health.harvard.edu":0.95,
    "medicalnewstoday.com":0.85,
    "healthline.com":0.85,
    "webmd.com":0.85,
    "news-medical.net":0.85,

    # =========================
    # 📰 TOP NEWS (HIGH TRUST)
    # =========================
    "bbc.com":0.95,
    "bbc.co.uk":0.95,
    "nytimes.com":0.95,
    "theguardian.com":0.95,
    "washingtonpost.com":0.95,
    "wsj.com":0.95,
    "bloomberg.com":0.95,
    "economist.com":0.95,

    # =========================
    # 🌍 GLOBAL MEDIA
    # =========================
    "cnn.com":0.9,
    "abcnews.go.com":0.9,
    "cbsnews.com":0.9,
    "nbcnews.com":0.9,
    "forbes.com":0.9,
    "time.com":0.9,

    # =========================
    # 🇮🇳 INDIAN MEDIA
    # =========================
    "thehindu.com":0.9,
    "indianexpress.com":0.9,
    "hindustantimes.com":0.85,
    "ndtv.com":0.85,
    "indiatoday.in":0.85,
    "timesofindia.indiatimes.com":0.8,
    "economictimes.indiatimes.com":0.8,

    # =========================
    # 🧠 TECH / DATA / AI
    # =========================
    "arxiv.org":0.9,
    "kaggle.com":0.8,
    "towardsdatascience.com":0.75,
    "medium.com":0.7,

    # =========================
    # ⚠️ LOW / MIXED TRUST
    # =========================
    "wikipedia.org":0.7,
    "quora.com":0.5,
    "reddit.com":0.5,
    "yahoo.com":0.6,
    "msn.com":0.6,

    # =========================
    # ❌ LOW CREDIBILITY / BLOGS
    # =========================
    "naturalnews.com":0.2,
    "beforeitsnews.com":0.2,
    "infowars.com":0.1,
    "clickbait.com":0.2

        }

    # ✅ Extract domain cleanly
    def extract_domain(self, url):

        try:
            domain = urlparse(url).netloc.lower()
            domain = domain.replace("www.", "")
            return domain
        except:
            return ""

    # ✅ Assign credibility score
    def get_credibility(self, domain):

        for d, score in self.credible_domains.items():
            if d in domain:
                return score

        return 0.3  # default low

    # ✅ Detect source type (optional but useful)
    def get_source_type(self, domain):

        if ".gov" in domain or ".edu" in domain:
            return "HIGH_TRUST"

        if any(d in domain for d in ["reuters","bbc","apnews"]):
            return "NEWS"

        if any(d in domain for d in ["snopes","factcheck","politifact"]):
            return "FACT_CHECK"

        return "UNKNOWN"

    # ✅ MAIN SEARCH FUNCTION
    def search(self, query):

        evidence = []

        if query.strip() == "":
            return evidence

        try:
            with DDGS() as ddgs:

                results = ddgs.text(query, max_results=10)

                for r in results:

                    url = r.get("href", "")
                    title = r.get("title", "")
                    snippet = r.get("body", "")

                    domain = self.extract_domain(url)

                    credibility = self.get_credibility(domain)

                    # ✅ FILTER LOW QUALITY
                    if credibility < 0.6:
                        continue

                    evidence.append({
                        "title": title,
                        "snippet": snippet,
                        "url": url,
                        "domain": domain,
                        "credibility_score": credibility,
                        "source_type": self.get_source_type(domain)
                    })

        except Exception as e:
            print("Search Error:", e)

        # ✅ REMOVE DUPLICATES
        unique = []
        seen = set()

        for e in evidence:
            if e["url"] not in seen:
                unique.append(e)
                seen.add(e["url"])

        return unique

class EvidenceRanker:

    def __init__(self):

        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

    def rank(self, claim, evidence):

        if not evidence:
            return []

        claim_embedding = self.encoder.encode(claim, convert_to_tensor=True)

        snippets = [e["snippet"] for e in evidence]

        snippet_embeddings = self.encoder.encode(snippets, convert_to_tensor=True)

        similarities = util.cos_sim(claim_embedding, snippet_embeddings)[0]

        for i,e in enumerate(evidence):

            semantic_score = float(similarities[i])

            e["score"] = 0.7*semantic_score + 0.3*e["credibility_score"]

        ranked = sorted(evidence, key=lambda x:x["score"], reverse=True)

        return ranked[:5]

class ClaimVerifier:

    def __init__(self):

        model_name = "mistralai/Mistral-7B-Instruct-v0.2"

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    def verify(self, claim, evidence):

        # ---------- Handle empty evidence ----------
        if len(evidence) == 0:
            return "FAKE", 0.3, "No credible evidence found."

        # ---------- Prepare evidence ----------
        top_evidence = evidence[:3]
        evidence_text = "\n".join([e["snippet"] for e in top_evidence])

        # ---------- Improved Prompt ----------
        prompt = f"""
You are a fact-checking assistant.

Claim:
{claim}

Evidence:
{evidence_text}

Task:
- Check whether evidence supports or contradicts the claim
- Use only given evidence

Respond EXACTLY like this:

VERDICT: TRUE or FALSE
CONFIDENCE: number between 0 and 1
EXPLANATION: short explanation based on evidence
"""

        # ---------- Tokenize ----------
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # ---------- Generate ----------
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.0,
                pad_token_id=self.tokenizer.eos_token_id
            )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # ---------- Default values ----------
        verdict = "REAL"
        confidence = 0.5
        explanation = "Could not determine clearly."

        # ---------- Safe Parsing ----------
        for line in response.split("\n"):

            line_upper = line.upper()

            if "VERDICT" in line_upper:
                if "FALSE" in line_upper:
                    verdict = "FAKE"
                elif "TRUE" in line_upper:
                    verdict = "REAL"

            elif "CONFIDENCE" in line_upper:
                import re
                nums = re.findall(r"\d*\.?\d+", line)
                if nums:
                    confidence = float(nums[0])

            elif "EXPLANATION" in line_upper:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    explanation = parts[1].strip()

        # ---------- Add Source-Based Explanation ----------
        high_cred_sources = [e for e in evidence if e["credibility_score"] >= 0.9]
        domains = [e["domain"] for e in top_evidence if "domain" in e]

        if verdict == "REAL":
            explanation += f" Supported by {len(high_cred_sources)} high-credibility sources."
        else:
            explanation += f" Contradicted by credible sources."

        if domains:
            explanation += f" Sources include: {', '.join(domains[:2])}."

        return verdict, confidence, explanation

class FakeNewsPipeline:

    def __init__(self):

        self.extractor = ClaimExtractor()
        self.query_gen = QueryGenerator()
        self.retriever = KnowledgeRetriever()
        self.ranker = EvidenceRanker()
        self.verifier = ClaimVerifier()

    def detect(self, article, verbose=True):

        claims = self.extractor.extract_claims(article)

        all_results = []

        for claim in claims:

            queries = self.query_gen.generate_queries([claim])

            evidence = []

            for q in queries:
                evidence.extend(self.retriever.search(q))

            ranked = self.ranker.rank(claim, evidence)

            verdict, conf, _ = self.verifier.verify(claim, ranked)

            all_results.append({"verdict": verdict, "confidence": conf})

        fake = sum(1 for r in all_results if r["verdict"] == "FAKE")
        real = sum(1 for r in all_results if r["verdict"] == "REAL")
        total = len(all_results)
        # ✅ Majority-based decision
        fake_ratio = fake / total
        real_ratio = real / total

        # ✅ Confidence-weighted decision (stronger)
        fake_score = sum(r["confidence"] for r in all_results if r["verdict"] == "FAKE")
        real_score = sum(r["confidence"] for r in all_results if r["verdict"] == "REAL")
        if fake_score > real_score:
            final_verdict = "FAKE"
        else:
            final_verdict = "REAL"
        avg_conf = sum(r["confidence"] for r in all_results) / total
        avg_conf = min(avg_conf, 0.85)

        if verbose:
            print("Verdict:", final_verdict, "| Confidence:", round(avg_conf,2))

        return {"verdict": final_verdict, "confidence": avg_conf}

In [6]:
import pandas as pd

# Load JSONL file
import json

data = []

with open("/kaggle/input/datasets/sonavp/fever-dataset/train.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

fever_df = pd.DataFrame(data)

# Keep required columns
fever_df = fever_df[["claim", "label"]]

# Remove NEI
fever_df = fever_df[fever_df["label"] != "NOT ENOUGH INFO"]

# Convert labels
def convert_label(x):
    if x == "SUPPORTS":
        return "REAL"
    else:
        return "FAKE"

fever_df["label"] = fever_df["label"].apply(convert_label)

# Rename for pipeline
fever_df = fever_df.rename(columns={"claim": "text"})

print(fever_df.head())
print("Total samples:", len(fever_df))

                                                text label
0                                                     FAKE
1  Michael Folivi competed with ten teams from 20...  FAKE
2  Asiatic Society of Bangladesh(housed in Nimtal...  REAL
3  Lindfield railway station has 3 bus routes, in...  REAL
4  Mukaradeeb('Wolf's Den') is a city in Iraq nea...  REAL
Total samples: 69051


In [10]:
# Use small sample (LLM is slow)
fever_sample = fever_df.sample(800, random_state=42)

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
pipeline = FakeNewsPipeline()

y_true = []
y_pred = []

for i, row in fever_sample.iterrows():

    article = row["text"]
    true_label = row["label"]

    try:
        result = pipeline.detect(article)

        if result is None:
            continue

        predicted = result["verdict"]

        # Handle PARTIALLY FAKE → treat as FAKE
        if predicted == "PARTIALLY FAKE":
            predicted = "FAKE"

        # Skip UNVERIFIED
        if predicted not in ["REAL", "FAKE"]:
            continue

        y_true.append(true_label)
        y_pred.append(predicted)

    except Exception as e:
        continue

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verdict: FAKE | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: FAKE | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: FAKE | Confidence: 0.3
Verdict: FAKE | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: FAKE | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: FAKE | Confidence: 0.1
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REAL | Confidence: 0.85
Verdict: REA

In [13]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, average="binary", pos_label="REAL")
recall = recall_score(y_true, y_pred, average="binary", pos_label="REAL")
f1 = f1_score(y_true, y_pred, average="binary", pos_label="REAL")

print("\n==============================")
print("MODEL EVALUATION (FEVER)")
print("==============================")

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

print("\nDetailed Report:\n")
print(classification_report(y_true, y_pred))


MODEL EVALUATION (FEVER)
Accuracy: 0.655
Precision: 0.6832
Recall: 0.7836
F1 Score: 0.7299

Detailed Report:

              precision    recall  f1-score   support

        FAKE       0.59      0.47      0.52       324
        REAL       0.68      0.78      0.73       476

    accuracy                           0.66       800
   macro avg       0.64      0.62      0.63       800
weighted avg       0.65      0.66      0.65       800

